Notebook này dùng để retrieve nguồn từ Chroma và dùng OpenAI Responses API để trả lời có citation.

# **1. Cài đặt thư viện**

In [ ]:
!pip install -q openai chromadb pandas

# **2. Import thư viện**

In [ ]:
import os
import json
import re
import unicodedata
from typing import Any, Dict, List, Optional

import chromadb
from openai import OpenAI

# **3. Khai báo cấu hình**

In [ ]:
CHROMA_DIR = "./chroma_ou_rag_db_openai"
COLLECTION_NAME = "ou_academic_rag_openai"
EMBEDDING_MODEL = "text-embedding-3-small"
GENERATION_MODEL = "gpt-4.1-mini"
TOP_K = 5

print("COLLECTION_NAME:", COLLECTION_NAME)
print("EMBEDDING_MODEL:", EMBEDDING_MODEL)
print("GENERATION_MODEL:", GENERATION_MODEL)

# **4. Khởi tạo client và collection**

In [ ]:
import os
from dotenv import load_dotenv

# Đọc biến môi trường từ file .env trong cùng thư mục project
load_dotenv()

if os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY đã sẵn sàng.")
else:
    print("Bạn chưa set OPENAI_API_KEY.")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = chroma_client.get_collection(name=COLLECTION_NAME)
print("Số item:", collection.count())

# **5. Hàm embedding query và retrieve**

In [ ]:
def embed_query(query: str) -> List[float]:
    """Embed câu hỏi bằng OpenAI."""
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=query)
    return response.data[0].embedding


def retrieve(query: str, top_k: int = 5, where: Dict[str, Any] = None) -> List[Dict[str, Any]]:
    """Retrieve top-k chunks từ Chroma."""
    query_embedding = embed_query(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for i in range(len(results["ids"][0])):
        hits.append({
            "rank": i + 1,
            "id": results["ids"][0][i],
            "distance": results["distances"][0][i],
            "document": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
        })
    return hits

def infer_where_filter(query: str) -> Dict[str, Any] | None:
    """
    Tự suy luận document_type cần ưu tiên dựa trên nội dung câu hỏi.
    Nếu không chắc, trả về None để tìm toàn bộ DB.
    """
    q = query.lower()

    # ============================================================
    # 1. Học phí
    # ============================================================
    if any(k in q for k in ["học phí", "mức phí", "đóng phí", "miễn giảm học phí"]):
        return {"document_type": "tuition"}

    # ============================================================
    # 2. Quy chế học vụ
    # Các câu về điều kiện tốt nghiệp, cảnh báo, bảo lưu, buộc thôi học
    # nên tìm trong toàn bộ regulation + student_handbook.
    # Chroma không hỗ trợ OR đơn giản giữa 2 document_type trong bản code này,
    # nên để None để hệ thống tìm toàn DB rồi rerank bằng keyword.
    # ============================================================
    if any(k in q for k in [
        "điều kiện tốt nghiệp",
        "điều kiện gì để được xét tốt nghiệp",
        "được xét tốt nghiệp",
        "công nhận tốt nghiệp",
        "cấp bằng tốt nghiệp",
        "cảnh báo học tập",
        "bảo lưu kết quả học tập",
        "nghỉ học tạm thời",
        "buộc thôi học",
        "buộc nghỉ học",
        "thi hộ",
        "nhờ người thi hộ",
        "xử lý vi phạm"
    ]):
        return None

    # ============================================================
    # 3. Chương trình đào tạo / PLO / tín chỉ / học phần
    # ============================================================
    if any(k in q for k in [
        "chương trình đào tạo", "ctđt", "chuẩn đầu ra", "plo",
        "tín chỉ", "học phần", "mã môn học", "điều kiện tiên quyết",
        "công nghệ thông tin"
    ]):
        return {"document_type": "curriculum"}

    # ============================================================
    # 4. Kế hoạch đào tạo năm học
    # Chỉ route academic_plan khi câu hỏi hỏi về mốc thời gian/kế hoạch/đợt.
    # ============================================================
    if any(k in q for k in [
        "kế hoạch đào tạo", "năm học", "học kỳ",
        "mốc thời gian", "đợt", "thời gian học tập",
        "thời gian công bố", "thời gian nhận đơn",
        "kế hoạch xét tốt nghiệp"
    ]):
        return {"document_type": "academic_plan"}

    # ============================================================
    # 5. Sổ tay / phòng ban / liên hệ / thủ tục
    # ============================================================
    if any(k in q for k in [
        "liên hệ", "phòng", "đơn vị", "thủ tục", "xác nhận",
        "vay vốn", "thời khóa biểu", "đăng ký môn học",
        "kết quả học tập cá nhân", "hệ thống nào"
    ]):
        return {"document_type": "student_handbook"}

    return None

def normalize_text(text: str) -> str:
    """
    Chuẩn hóa text để so khớp dễ hơn:
    - chuyển về chữ thường
    - bỏ dấu tiếng Việt
    - thay ký tự đặc biệt bằng khoảng trắng
    - gom nhiều khoảng trắng thành 1
    """
    if text is None:
        return ""

    text = str(text).lower().strip()

    # Bỏ dấu tiếng Việt
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")

    # Chuẩn hóa đ/Đ
    text = text.replace("đ", "d")

    # Thay ký tự đặc biệt bằng khoảng trắng
    text = re.sub(r"[^a-z0-9]+", " ", text)

    # Gom nhiều khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


def compact_text(text: str) -> str:
    """
    Tạo dạng text không khoảng trắng để match tên file.
    Ví dụ:
    'Công nghệ thông tin' -> 'congnghethongtin'
    """
    return normalize_text(text).replace(" ", "")

def detect_major_from_query(query: str) -> Optional[Dict[str, Any]]:
    """
    Phát hiện ngành được nhắc trong câu hỏi.

    Trả về dict gồm:
    - name: tên ngành chuẩn để debug/hiển thị
    - triggers: các cụm có thể xuất hiện trong câu hỏi
    - keywords: từ khóa dùng để kiểm tra trong nội dung chunk
    - filename_keywords: từ khóa dùng để kiểm tra trong document_name

    Hàm này đã chuẩn hóa tiếng Việt nên match được cả có dấu và không dấu.
    """

    q_norm = normalize_text(query)
    q_compact = compact_text(query)

    major_catalog = [
        # ======================================================
        # Nhóm Công nghệ thông tin / máy tính
        # ======================================================
        {
            "name": "Công nghệ thông tin",
            "triggers": ["công nghệ thông tin", "cong nghe thong tin", "cntt", "information technology", "information technologies"],
            "keywords": ["công nghệ thông tin", "cong nghe thong tin", "cntt"],
            "filename_keywords": ["congnghethongtin", "cntt", "informationtechnology", "informationtechnologies"],
        },
        {
            "name": "Khoa học máy tính",
            "triggers": ["khoa học máy tính", "khoa hoc may tinh", "computer science"],
            "keywords": ["khoa học máy tính", "khoa hoc may tinh", "computer science"],
            "filename_keywords": ["khoahocmaytinh", "computerscience"],
        },
        {
            "name": "Khoa học dữ liệu",
            "triggers": ["khoa học dữ liệu", "khoa hoc du lieu", "data science"],
            "keywords": ["khoa học dữ liệu", "khoa hoc du lieu", "data science"],
            "filename_keywords": ["khoahocdulieu", "datascience"],
        },
        {
            "name": "Trí tuệ nhân tạo",
            "triggers": ["trí tuệ nhân tạo", "tri tue nhan tao", "artificial intelligence", "ai"],
            "keywords": ["trí tuệ nhân tạo", "tri tue nhan tao", "artificial intelligence"],
            "filename_keywords": ["trituenhan tao", "trituenhan", "trituenhan tao", "trituenhantao", "artificialintelligence", "ai"],
        },
        {
            "name": "Hệ thống thông tin quản lý",
            "triggers": ["hệ thống thông tin quản lý", "he thong thong tin quan ly", "management information system", "mis"],
            "keywords": ["hệ thống thông tin quản lý", "he thong thong tin quan ly", "management information system"],
            "filename_keywords": ["hethongthongtinquanly", "managementinformationsystem", "mis"],
        },
        {
            "name": "Kỹ thuật phần mềm",
            "triggers": ["kỹ thuật phần mềm", "ky thuat phan mem", "software engineering"],
            "keywords": ["kỹ thuật phần mềm", "ky thuat phan mem", "software engineering"],
            "filename_keywords": ["kythuatphanmem", "softwareengineering"],
        },
        {
            "name": "An toàn thông tin",
            "triggers": ["an toàn thông tin", "an toan thong tin", "information security"],
            "keywords": ["an toàn thông tin", "an toan thong tin", "information security"],
            "filename_keywords": ["antoanthongtin", "informationsecurity"],
        },

        # ======================================================
        # Nhóm Kinh tế - quản trị - tài chính
        # ======================================================
        {
            "name": "Kế toán",
            "triggers": ["kế toán", "ke toan", "accounting"],
            "keywords": ["kế toán", "ke toan", "accounting"],
            "filename_keywords": ["ketoan", "accounting"],
        },
        {
            "name": "Kiểm toán",
            "triggers": ["kiểm toán", "kiem toan", "auditing", "audit"],
            "keywords": ["kiểm toán", "kiem toan", "auditing", "audit"],
            "filename_keywords": ["kiemtoan", "auditing", "audit"],
        },
        {
            "name": "Tài chính - Ngân hàng",
            "triggers": ["tài chính ngân hàng", "tài chính - ngân hàng", "tai chinh ngan hang", "finance banking", "banking finance"],
            "keywords": ["tài chính ngân hàng", "tai chinh ngan hang", "finance", "banking"],
            "filename_keywords": ["taichinhnganhang", "financebanking", "bankingfinance"],
        },
        {
            "name": "Quản trị kinh doanh",
            "triggers": ["quản trị kinh doanh", "quan tri kinh doanh", "business administration"],
            "keywords": ["quản trị kinh doanh", "quan tri kinh doanh", "business administration"],
            "filename_keywords": ["quantrikinhdoanh", "businessadministration"],
        },
        {
            "name": "Marketing",
            "triggers": ["marketing", "tiếp thị", "tiep thi"],
            "keywords": ["marketing", "tiếp thị", "tiep thi"],
            "filename_keywords": ["marketing", "tiepthi"],
        },
        {
            "name": "Kinh doanh quốc tế",
            "triggers": ["kinh doanh quốc tế", "kinh doanh quoc te", "international business"],
            "keywords": ["kinh doanh quốc tế", "kinh doanh quoc te", "international business"],
            "filename_keywords": ["kinhdoanhquocte", "internationalbusiness"],
        },
        {
            "name": "Quản trị nhân lực",
            "triggers": ["quản trị nhân lực", "quan tri nhan luc", "human resource management", "hrm"],
            "keywords": ["quản trị nhân lực", "quan tri nhan luc", "human resource management"],
            "filename_keywords": ["quantrilhanluc", "quantrinhansul", "quantrinhansu", "quantrinhannluc", "quantrinhânluc", "quantrinhânlực", "quantrinhannluc", "humanresourcemanagement", "hrm"],
        },

        # ======================================================
        # Nhóm Luật
        # ======================================================
        {
            "name": "Luật",
            "triggers": ["ngành luật", "luật học", "luat hoc", "law"],
            "keywords": ["ngành luật", "luật", "luat", "law"],
            "filename_keywords": ["luat", "law"],
        },
        {
            "name": "Luật kinh tế",
            "triggers": ["luật kinh tế", "luat kinh te", "economic law"],
            "keywords": ["luật kinh tế", "luat kinh te", "economic law"],
            "filename_keywords": ["luatkinhte", "economiclaw"],
        },

        # ======================================================
        # Nhóm Ngôn ngữ
        # ======================================================
        {
            "name": "Ngôn ngữ Anh",
            "triggers": ["ngôn ngữ anh", "ngon ngu anh", "english language"],
            "keywords": ["ngôn ngữ anh", "ngon ngu anh", "english language"],
            "filename_keywords": ["ngonnguanh", "englishlanguage"],
        },
        {
            "name": "Ngôn ngữ Nhật",
            "triggers": ["ngôn ngữ nhật", "ngon ngu nhat", "japanese language"],
            "keywords": ["ngôn ngữ nhật", "ngon ngu nhat", "japanese language"],
            "filename_keywords": ["ngonngunhat", "japaneselanguage"],
        },
        {
            "name": "Ngôn ngữ Trung Quốc",
            "triggers": ["ngôn ngữ trung quốc", "ngon ngu trung quoc", "chinese language"],
            "keywords": ["ngôn ngữ trung quốc", "ngon ngu trung quoc", "chinese language"],
            "filename_keywords": ["ngonngutrungquoc", "chineselanguage"],
        },

        # ======================================================
        # Nhóm Công nghệ - xây dựng - sinh học
        # ======================================================
        {
            "name": "Công nghệ sinh học",
            "triggers": ["công nghệ sinh học", "cong nghe sinh hoc", "biotechnology"],
            "keywords": ["công nghệ sinh học", "cong nghe sinh hoc", "biotechnology"],
            "filename_keywords": ["congnghesinhhoc", "biotechnology"],
        },
        {
            "name": "Công nghệ thực phẩm",
            "triggers": ["công nghệ thực phẩm", "cong nghe thuc pham", "food technology"],
            "keywords": ["công nghệ thực phẩm", "cong nghe thuc pham", "food technology"],
            "filename_keywords": ["congnghethucpham", "foodtechnology"],
        },
        {
            "name": "Sinh học ứng dụng",
            "triggers": ["sinh học ứng dụng", "sinh hoc ung dung", "applied biology"],
            "keywords": ["sinh học ứng dụng", "sinh hoc ung dung", "applied biology"],
            "filename_keywords": ["sinhhocungdung", "appliedbiology"],
        },
        {
            "name": "Công nghệ kỹ thuật công trình xây dựng",
            "triggers": ["công nghệ kỹ thuật công trình xây dựng", "cong nghe ky thuat cong trinh xay dung"],
            "keywords": ["công nghệ kỹ thuật công trình xây dựng", "cong nghe ky thuat cong trinh xay dung"],
            "filename_keywords": ["congnghekythuatcongtrinhxaydung"],
        },
        {
            "name": "Quản lý xây dựng",
            "triggers": ["quản lý xây dựng", "quan ly xay dung", "construction management"],
            "keywords": ["quản lý xây dựng", "quan ly xay dung", "construction management"],
            "filename_keywords": ["quanlyxaydung", "constructionmanagement"],
        },
        {
            "name": "Kiến trúc",
            "triggers": ["kiến trúc", "kien truc", "architecture"],
            "keywords": ["kiến trúc", "kien truc", "architecture"],
            "filename_keywords": ["kientruc", "architecture"],
        },
        {
            "name": "Kỹ thuật xây dựng",
            "triggers": ["kỹ thuật xây dựng", "ky thuat xay dung", "civil engineering"],
            "keywords": ["kỹ thuật xây dựng", "ky thuat xay dung", "civil engineering"],
            "filename_keywords": ["kythuatxaydung", "civilengineering"],
        },

        # ======================================================
        # Nhóm khác
        # ======================================================
        {
            "name": "Du lịch",
            "triggers": ["du lịch", "du lich", "tourism"],
            "keywords": ["du lịch", "du lich", "tourism"],
            "filename_keywords": ["dulich", "tourism"],
        },
        {
            "name": "Xã hội học",
            "triggers": ["xã hội học", "xa hoi hoc", "sociology"],
            "keywords": ["xã hội học", "xa hoi hoc", "sociology"],
            "filename_keywords": ["xahoihoc", "sociology"],
        },
        {
            "name": "Đông Nam Á học",
            "triggers": ["đông nam á học", "dong nam a hoc", "southeast asian studies"],
            "keywords": ["đông nam á học", "dong nam a hoc", "southeast asian studies"],
            "filename_keywords": ["dongnamahoc", "southeastasianstudies"],
        },
        {
            "name": "Công tác xã hội",
            "triggers": ["công tác xã hội", "cong tac xa hoi", "social work"],
            "keywords": ["công tác xã hội", "cong tac xa hoi", "social work"],
            "filename_keywords": ["congtacxahoi", "socialwork"],
        },
    ]

    # Ưu tiên match cụm dài trước để tránh nhầm:
    # Ví dụ "Luật kinh tế" phải được nhận trước "Luật".
    major_catalog = sorted(
        major_catalog,
        key=lambda m: max(len(normalize_text(t)) for t in m["triggers"]),
        reverse=True
    )

    for major in major_catalog:
        trigger_norms = [normalize_text(t) for t in major["triggers"]]
        trigger_compacts = [compact_text(t) for t in major["triggers"]]

        # Match dạng có khoảng trắng
        if any(t in q_norm for t in trigger_norms):
            return major

        # Match dạng dính liền, phòng trường hợp tên file/câu hỏi bị mất khoảng trắng
        if any(t in q_compact for t in trigger_compacts):
            return major

    return None

def keyword_boost_score(query: str, hit: Dict[str, Any]) -> float:
    """
    Tính điểm boost thủ công để rerank lại kết quả retrieval.
    Chroma trả về distance càng nhỏ càng tốt.
    Ta trừ boost khỏi distance để chunk phù hợp được đẩy lên cao hơn.
    """
    q = query.lower()
    text = (hit.get("document") or "").lower()
    meta = hit.get("metadata") or {}

    doc_name = str(meta.get("document_name", "")).lower()
    section = str(meta.get("section", "")).lower()
    chunk_type = str(meta.get("chunk_type", "")).lower()
    doc_type = str(meta.get("document_type", "")).lower()

    # Bản chuẩn hóa không dấu
    q_norm = normalize_text(query)
    text_norm = normalize_text(text)
    doc_name_norm = normalize_text(doc_name)
    doc_name_compact = compact_text(doc_name)

    boost = 0.0

    # ============================================================
    # Major-aware boost: ưu tiên đúng ngành được hỏi
    # ============================================================
    major = detect_major_from_query(query)

    if major is not None:
        filename_keywords = [compact_text(k) for k in major["filename_keywords"]]
        text_keywords = [normalize_text(k) for k in major["keywords"]]

        # 1. document_name đúng ngành là tín hiệu mạnh nhất.
        if any(k in doc_name_compact for k in filename_keywords):
            boost += 14.0

        # 2. Nội dung chunk có nhắc đúng ngành.
        if any(k in text_norm for k in text_keywords):
            boost += 5.0

        # 3. Nếu đang hỏi curriculum/PLO mà chunk không cùng ngành, phạt nhẹ.
        # Điều này giúp giảm tình trạng hỏi Kế toán nhưng ra Luật/CNTT/HTTTQL.
        is_same_major = (
            any(k in doc_name_compact for k in filename_keywords)
            or any(k in text_norm for k in text_keywords)
        )

        if doc_type == "curriculum" and not is_same_major:
            boost -= 5.0

    # Câu hỏi về PLO
    if "plo" in q or "chuẩn đầu ra" in q:
        if chunk_type == "plo":
            boost += 8.0
        if "công nghệ thông tin" in text:
            boost += 2.0

    # Câu hỏi về học phí
    if "học phí" in q or "mức phí" in q:
        if doc_type == "tuition":
            boost += 10.0
        if "chương trình chuẩn" in q and "chương trình chuẩn" in text:
            boost += 3.0
        if "công nghệ thông tin" in text:
            boost += 5.0

    # Câu hỏi về buộc thôi học
    if "buộc thôi học" in q:
        if "2. buộc thôi học" in section or "## 2. buộc thôi học" in text:
            boost += 7.0
        elif "buộc thôi học" in text:
            boost += 3.0

    # Câu hỏi kế hoạch
    if "kế hoạch" in q or "năm học" in q:
        if doc_type == "academic_plan":
            boost += 4.0
        if "xét tốt nghiệp" in q and "kế hoạch xét tốt nghiệp" in text:
            boost += 4.0

    return boost

    # Câu hỏi điều kiện xét/công nhận tốt nghiệp
    if any(k in q for k in ["điều kiện tốt nghiệp", "được xét tốt nghiệp", "công nhận tốt nghiệp", "cấp bằng tốt nghiệp"]):
        if "điều 26" in text or "công nhận tốt nghiệp và cấp bằng tốt nghiệp" in text:
            boost += 8.0
        if doc_type in ["regulation", "student_handbook"]:
            boost += 3.0

def get_adaptive_retrieval_params(query: str) -> Dict[str, int]:
    """
    Tự điều chỉnh số nguồn lấy ra theo loại câu hỏi.
    """
    q = query.lower()

    if "plo" in q or "chuẩn đầu ra" in q.lower():
        return {
            "top_k": 10,
            "raw_top_k": 50,
            "preview_chars": 1600
        }

    if "học phí" in q or "mức phí" in q:
        return {
            "top_k": 5,
            "raw_top_k": 20,
            "preview_chars": 1500
        }

    return {
        "top_k": 5,
        "raw_top_k": 20,
        "preview_chars": 900
    }


def retrieve_routed(
    query: str,
    top_k: int = 5,
    raw_top_k: int = 20,
    where: Dict[str, Any] | None = None
) -> List[Dict[str, Any]]:
    """
    Retrieval cải tiến:
    1. Tự suy luận filter document_type nếu người dùng không truyền where.
    2. Lấy raw_top_k kết quả ban đầu.
    3. Rerank bằng keyword_boost_score.
    4. Trả về top_k kết quả tốt nhất.
    """
    auto_where = where if where is not None else infer_where_filter(query)

    raw_hits = retrieve(
        query=query,
        top_k=raw_top_k,
        where=auto_where
    )

    reranked = []
    for hit in raw_hits:
        boost = keyword_boost_score(query, hit)
        final_score = float(hit["distance"]) - 0.05 * boost

        hit["auto_where"] = auto_where
        hit["keyword_boost"] = boost
        hit["final_score"] = final_score

        reranked.append(hit)

    reranked.sort(key=lambda x: x["final_score"])

    final_hits = reranked[:top_k]

    for i, hit in enumerate(final_hits, start=1):
        hit["rank"] = i

    return final_hits

# **6. Hàm format nguồn**

In [ ]:
def format_source(meta: Dict[str, Any]) -> str:
    """
    Format metadata thành nguồn ngắn gọn, KHÔNG hiển thị tên file gốc.
    Mục tiêu:
    - Người dùng thấy nguồn dễ đọc.
    - Không lộ tên file markdown/PDF dài.
    - Vẫn giữ citation dạng [NGUỒN 1], [NGUỒN 2].
    """

    document_type = meta.get("document_type", "")
    chunk_type = meta.get("chunk_type", "")
    section = meta.get("section", "")
    article = meta.get("article", "")
    page_start = meta.get("page_start", "")
    page_end = meta.get("page_end", "")

    # Map document_type sang tên dễ hiểu cho người dùng.
    doc_type_map = {
        "regulation": "Quy chế đào tạo",
        "student_handbook": "Sổ tay sinh viên",
        "curriculum": "Chương trình đào tạo",
        "academic_plan": "Kế hoạch đào tạo năm học",
        "tuition": "Thông tin học phí",
    }

    source_name = doc_type_map.get(document_type, "Tài liệu học vụ")

    parts = [source_name]

    if section not in [None, "", "null"]:
        parts.append(f"Mục: {section}")

    if article not in [None, "", "null"]:
        parts.append(f"Điều: {article}")

    # Chỉ hiện trang nếu có thông tin trang hợp lệ.
    if page_start not in [None, "", "null"] and page_end not in [None, "", "null"]:
        if page_start == page_end:
            parts.append(f"Trang: {page_start}")
        else:
            parts.append(f"Trang: {page_start}-{page_end}")

    return " | ".join(parts)

# **7. Hàm build context**

In [ ]:
def build_context(hits: List[Dict[str, Any]], max_chars_per_chunk: int = 3500) -> str:
    """Ghép các chunks thành context có đánh số [NGUỒN 1], [NGUỒN 2],..."""
    blocks = []
    for i, hit in enumerate(hits, start=1):
        source = format_source(hit["metadata"])
        text = hit["document"][:max_chars_per_chunk]
        blocks.append(f"""
[NGUỒN {i}]
{source}

{text}
""".strip())
    return "\n\n".join(blocks)

# **8. System prompt**

In [ ]:
SYSTEM_PROMPT = """
Bạn là chatbot RAG hỗ trợ sinh viên tra cứu thông tin học vụ, quy chế đào tạo,
chương trình đào tạo, kế hoạch đào tạo và học phí của Trường Đại học Mở TP.HCM.

Nhiệm vụ của bạn là trả lời câu hỏi của sinh viên dựa trên các NGUỒN tài liệu
được truy xuất từ hệ thống retrieval. Mỗi nguồn sẽ được đánh số theo dạng
[NGUỒN 1], [NGUỒN 2], [NGUỒN 3], ...

============================================================
NGUYÊN TẮC BẮT BUỘC
============================================================

1. Chỉ sử dụng thông tin có trong các NGUỒN được cung cấp.
   Không tự suy đoán, không tự bổ sung thông tin ngoài tài liệu.

2. Nếu các nguồn không có đủ thông tin để trả lời chắc chắn, hãy nói rõ:
   "Mình chưa tìm thấy thông tin đủ chắc chắn trong tài liệu hiện có."
   Sau đó có thể gợi ý sinh viên liên hệ đơn vị phù hợp nếu nguồn có nhắc đến.

3. Không được bịa số liệu, mốc thời gian, mức học phí, điều kiện tốt nghiệp,
   chuẩn đầu ra, mã môn học, số tín chỉ hoặc tên phòng ban.

4. Khi trả lời một ý quan trọng, phải gắn citation ngay sau ý đó bằng dạng:
   [NGUỒN 1] hoặc [NGUỒN 1], [NGUỒN 2].

5. Cuối câu trả lời phải có mục:
   "Nguồn tham khảo:"
   và liệt kê ngắn gọn các nguồn đã dùng.

6. Nếu nhiều nguồn có nội dung trùng nhau, ưu tiên nguồn có nội dung trực tiếp
   và rõ ràng nhất. Không cần lặp lại cùng một thông tin nhiều lần.

7. Nếu câu hỏi hỏi về một ngành cụ thể, ví dụ Công nghệ thông tin, chỉ dùng nguồn
   đúng ngành đó. Nếu nguồn trả về thuộc ngành khác, không dùng để kết luận.

8. Nếu câu hỏi hỏi về học phí, cần phân biệt rõ:
   - chương trình chuẩn hay chương trình tiên tiến;
   - nhóm ngành/ngành được hỏi;
   - mức học phí là mức bình quân/dự kiến nếu tài liệu ghi như vậy.
   Không tự quy đổi sang học phí theo tín chỉ nếu tài liệu không nêu.

9. Nếu câu hỏi hỏi về PLO/chuẩn đầu ra, hãy tổng hợp các PLO tìm được từ nguồn.
   Nếu nguồn chỉ cung cấp một phần PLO, hãy nói rõ câu trả lời dựa trên các PLO
   được truy xuất trong nguồn hiện có.

10. Nếu câu hỏi hỏi về kế hoạch đào tạo hoặc mốc thời gian, phải nêu đúng năm học,
    học kỳ, đợt, ngày/tháng/năm nếu nguồn có ghi.

============================================================
CÁCH TRẢ LỜI
============================================================

- Trả lời bằng tiếng Việt.
- Văn phong rõ ràng, thân thiện, phù hợp với sinh viên.
- Ưu tiên trình bày theo gạch đầu dòng khi có nhiều điều kiện, nhiều mốc thời gian,
  nhiều học phần hoặc nhiều chuẩn đầu ra.
- Không trả lời quá dài nếu câu hỏi đơn giản.
- Không sao chép toàn bộ nguồn nếu không cần thiết; hãy tóm tắt đúng trọng tâm.
- Nếu có bảng trong nguồn, hãy diễn giải lại thành các ý dễ hiểu.
- Ưu tiên trả lời trực tiếp đúng câu hỏi.
- Không mở rộng sang các quy định liên quan khác nếu câu hỏi không yêu cầu.
- Nếu có nguồn nhiễu hoặc nguồn chỉ liên quan gián tiếp, không dùng nguồn đó để thêm ý phụ.

============================================================
CẤU TRÚC CÂU TRẢ LỜI KHUYẾN NGHỊ
============================================================

1. Trả lời trực tiếp câu hỏi trước.
2. Nêu chi tiết theo từng ý nếu cần.
3. Nếu có điều kiện/mốc thời gian/mức phí, trình bày rõ từng dòng.
4. Kết thúc bằng mục "Nguồn tham khảo".

Ví dụ format:

Theo tài liệu được truy xuất, ...

- Ý 1 ... [NGUỒN 1]
- Ý 2 ... [NGUỒN 1]
- Ý 3 ... [NGUỒN 2]

Nguồn tham khảo:
- [NGUỒN 1]&#58; ...
- [NGUỒN 2]&#58; ...

Lưu ý về citation:
- Không chỉ liệt kê nguồn ở cuối.
- Mỗi gạch đầu dòng hoặc mỗi ý quan trọng phải có citation ngay sau nội dung, ví dụ: [NGUỒN 1].
- Mục "Nguồn tham khảo" cuối câu trả lời phải ghi rõ nguồn đó là tài liệu/mục nào nếu metadata có cung cấp.

Lưu ý về nguồn tham khảo:
- Nếu nguồn giống nhau về nội dung, chỉ cần chọn 1 nguồn có nội dung rõ ràng nhất để trích dẫn.
- Không hiển thị tên file gốc như .md, .pdf, .docling hoặc đường dẫn file.
- Khi liệt kê nguồn, chỉ ghi loại tài liệu và mục/điều/trang nếu có.
- Ví dụ đúng:
  - [NGUỒN 1]&#58; Kế hoạch đào tạo năm học, Mục 11. Kế hoạch xét tốt nghiệp.
- Ví dụ không dùng:
  - [NGUỒN 1]&#58; CV_996-KH-ĐHM_07052025_...

====================================================================
XỬ LÝ TRƯỜNG HỢP KHÔNG ĐỦ THÔNG TIN
============================================================

Nếu nguồn không đủ thông tin, trả lời theo mẫu:

"Mình chưa tìm thấy thông tin đủ chắc chắn trong tài liệu hiện có để kết luận về nội dung này."

Nếu có nguồn liên quan nhưng chưa trực tiếp trả lời, hãy nói:

"Tài liệu có nhắc đến ... nhưng chưa nêu rõ ..."

Không được cố trả lời bằng kiến thức ngoài nguồn.
""".strip()

# **9. Hàm gọi OpenAI sinh câu trả lời**

In [ ]:
def generate_answer_with_openai(question: str, context: str) -> str:
    """Sinh câu trả lời bằng OpenAI Responses API."""
    user_prompt = f"""
CÂU HỎI CỦA SINH VIÊN:
{question}

CÁC NGUỒN TRUY XUẤT ĐƯỢC:
{context}

Yêu cầu:
- Trả lời trực tiếp vào câu hỏi.
- Nếu có nhiều điều kiện/quy định, trình bày theo từng ý.
- Dùng citation [NGUỒN 1], [NGUỒN 2] ngay sau ý liên quan.
- Không dùng kiến thức ngoài nguồn.
""".strip()

    response = client.responses.create(
        model=GENERATION_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )

    if hasattr(response, "output_text") and response.output_text:
        return response.output_text
    try:
        return response.output[0].content[0].text
    except Exception:
        return str(response)

# **10. Hàm RAG hoàn chỉnh**

In [ ]:
def answer_question(question: str, where: Dict[str, Any] = None) -> Dict[str, Any]:
    """
    Pipeline RAG hoàn chỉnh:
    question → retrieve_routed → build context → OpenAI answer.

    where:
    - None: hệ thống tự route theo câu hỏi.
    - Có giá trị: ép tìm trong document_type cụ thể.
    """
    params = get_adaptive_retrieval_params(question)

    hits = retrieve_routed(
        query=question,
        top_k=params["top_k"],
        raw_top_k=params["raw_top_k"],
        where=where
    )

    context = build_context(hits)
    answer = generate_answer_with_openai(question, context)

    return {
        "answer": answer,
        "hits": hits,
        "context": context,
        "params": params
    }

# **11. Test chatbot**

## **11.1. Câu hỏi test 1**

In [ ]:
question = "Sinh viên cần điều kiện gì để được xét tốt nghiệp?"
result = answer_question(question)

print("CÂU HỎI:")
print(question)
print("\nCÂU TRẢ LỜI:")
print(result["answer"])

## **11.2.  Debug nguồn câu hỏi 1**

In [ ]:
for hit in result["hits"]:
    print("=" * 100)
    print("RANK:", hit["rank"])
    print("DISTANCE:", hit["distance"])
    print("ID:", hit["id"])
    print("SOURCE:", format_source(hit["metadata"]))
    print("\nTEXT PREVIEW:")
    print(hit["document"][:1200])
    print()

## **11.3. Câu hỏi test 2**

In [ ]:
question = "Học phí được tính như thế nào?"
result = answer_question(question)
print("CÂU HỎI:")
print(question)
print("\nCÂU TRẢ LỜI:")
print(result["answer"])

## **11.4. Debug nguồn câu hỏi 2**

In [ ]:
for hit in result["hits"]:
    print("=" * 100)
    print("RANK:", hit["rank"])
    print("DISTANCE:", hit["distance"])
    print("ID:", hit["id"])
    print("SOURCE:", format_source(hit["metadata"]))
    print("\nTEXT PREVIEW:")
    print(hit["document"][:1200])
    print()